In [1]:
from internalizer import Internalizer
import pandas as pd

16:54:39+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
# fp = "/p/tmp/davidba/internalization/remind/output/SSP2-NPi-internalizeEI-coupled-run0_2025-02-04_17.41.53/REMIND_generic_SSP2-NPi-internalizeEI-coupled-run0.mif"
fp = "/p/tmp/davidba/internalizer/dev/remind_runs/remind_SSP2-NPi.mif"
ei_version = "3.10"
bw_project = f"internalizer_ei_{ei_version}"
years = [2020, 2030, 2040, 2050, 2060, 2070]

In [3]:
I = Internalizer(fp, "remind", "SSP2-NPi", ei_version, bw_project)

In [4]:
I.years = years

## Generate the mapping for FE

In [5]:
from internalizer.regionalization import REMIND_REGIONS

In [17]:
FEmapping = pd.read_csv("mappings/demFe.csv", sep=";")

In [18]:
dflist = []
for region in REMIND_REGIONS:
    df = FEmapping[FEmapping["share"] != "regional"].copy()
    df["region"] = region
    dflist.append(df)

In [21]:
regionalized_mapping = pd.concat(dflist, axis=0, ignore_index=True).rename(columns={"all_enty - emi_sectors": "REMIND index"})

In [22]:
regionalized_mapping

,REMIND index,dataset name,dataset reference product,dataset unit,share,region
0,fesos - indst,"hard coal, burned in hard coal industrial furn...",heat,megajoule,1.0,CAZ
1,fesos - build,"wood pellet, burned in residential stove 9kW",heat,megajoule,1.0,CAZ
2,fehos - indst,"light fuel oil, burned in industrial furnace 1MW",heat,megajoule,1.0,CAZ
3,fehos - build,"light fuel oil, burned in residential boiler 10kW",heat,megajoule,1.0,CAZ
4,fegas - indst,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,1.0,CAZ
...,...,...,...,...,...,...
259,feels - build,"market group for electricity, low voltage","electricity, low voltage",kilowatt hour,1.0,USA
260,feelt - trans,"electricity, used in passenger car","electricity, low voltage",kilowatt hour,1.0,USA
261,fepet - trans,"petrol, burned in passenger car",heat,megajoule,1.0,USA
262,fedie - trans,"diesel blend, burned in heavy-duty vehicle",heat,megajoule,1.0,USA


## Get list of extra activities

In [10]:
remove_activities = pd.read_csv("mappings/demFe_upstreams.csv", sep=";")
remove_activities

,all_enty,emi_sectors,dataset name,dataset reference product,dataset unit
0,fesos,indst,market for hard coal,hard coal,kilogram
1,fesos,build,"market for biomass, used as fuel","biomass, used as fuel",kilogram
2,fehos,indst,market for light fuel oil,light fuel oil,kilogram
3,fehos,build,market for light fuel oil,light fuel oil,kilogram
4,fegas,indst,"petroleum and gas production, offshore","natural gas, high pressure",cubic meter
5,fegas,indst,"petroleum and gas production, onshore","natural gas, high pressure",cubic meter
6,fegas,CDR,"petroleum and gas production, offshore","natural gas, high pressure",cubic meter
7,fegas,CDR,"petroleum and gas production, onshore","natural gas, high pressure",cubic meter
8,fegas,build,"market for natural gas, high pressure","natural gas, high pressure",cubic meter
9,fegat,trans,"market group for natural gas, high pressure","natural gas, high pressure",cubic meter


In [11]:
import numpy as np

In [12]:
q = np.arange(0.1, 1.0, 0.1)
q

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

In [14]:
I.calculate_costs_new(regionalized_mapping, q, True, remove_activities=remove_activities)

/p/tmp/davidba/internalizer/internalizer/regionalization.py:249: PerformanceWarning: indexing past lexsort depth may impact performance.
  sel = df.loc[i]
/p/tmp/davidba/internalizer/internalizer/regionalization.py:249: PerformanceWarning: indexing past lexsort depth may impact performance.
  sel = df.loc[i]
/p/tmp/davidba/internalizer/internalizer/regionalization.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf["region"] = region
/p/tmp/davidba/internalizer/internalizer/regionalization.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/inde

KeyError: 'REMIND tech'

In [15]:
regionalized_costs = pd.read_csv("output/remind_runs/remind/SSP2-NPi/2030/regionalized_costs.csv")
regionalized_costs

,dataset name,dataset reference product,dataset unit,region,quantile,impact category,cost
0,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,CAZ,0.1,acidification,2.145162e-04
1,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,CAZ,0.1,climate change,3.042344e-03
2,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,CAZ,0.1,ecotoxicity,7.454801e-05
3,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,CAZ,0.1,eutrophication,1.210335e-04
4,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,CAZ,0.1,fossil resources,3.269018e-04
...,...,...,...,...,...,...,...
9121,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,World,0.9,metal/mineral resources,7.313624e-05
9122,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,World,0.9,ozone depletion,7.410769e-07
9123,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,World,0.9,particulate matter formation,2.371633e-03
9124,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,World,0.9,photochemical oxidant formation,4.762747e-03


In [23]:
regionalized_mapping

,REMIND index,dataset name,dataset reference product,dataset unit,share,region
0,fesos - indst,"hard coal, burned in hard coal industrial furn...",heat,megajoule,1.0,CAZ
1,fesos - build,"wood pellet, burned in residential stove 9kW",heat,megajoule,1.0,CAZ
2,fehos - indst,"light fuel oil, burned in industrial furnace 1MW",heat,megajoule,1.0,CAZ
3,fehos - build,"light fuel oil, burned in residential boiler 10kW",heat,megajoule,1.0,CAZ
4,fegas - indst,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,1.0,CAZ
...,...,...,...,...,...,...
259,feels - build,"market group for electricity, low voltage","electricity, low voltage",kilowatt hour,1.0,USA
260,feelt - trans,"electricity, used in passenger car","electricity, low voltage",kilowatt hour,1.0,USA
261,fepet - trans,"petrol, burned in passenger car",heat,megajoule,1.0,USA
262,fedie - trans,"diesel blend, burned in heavy-duty vehicle",heat,megajoule,1.0,USA


In [25]:
def combine_shares_and_costs_new(shares: pd.DataFrame, costs: pd.DataFrame) -> pd.DataFrame:
    """
    Weight costs per shares.
    :param shares: dataframe of regionalized shares
    :param costs: dataframe of regionalized costs
    :return: regionalized aggregated costs per REMIND technology
    """
    shares = shares.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])
    costs = costs.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])

    dflist = []
    costs_index = costs.index
    for idx, row in shares.iterrows():
        tech = row["REMIND index"]
        factor = row["share"]
        j = idx
        if idx not in costs_index:
            j = (idx[0], idx[1], idx[2], "World")
        try:
            sel = costs.loc[j]
        except KeyError:
            break
        sel["cost"] = factor * sel["cost"]
        sel = sel.pivot(index="quantile", columns="impact category", values="cost").reset_index()
        sel["REMIND index"] = tech
        sel["region"] = idx[-1]
        dflist.append(sel)
        
    return pd.concat(dflist, axis=0, ignore_index=True).groupby(["REMIND index", "region", "quantile"]).sum()

In [36]:
shares = regionalized_mapping.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])
costs = regionalized_costs.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])

dflist = []
costs_index = costs.index
for idx, row in shares.iterrows():
    tech = row["REMIND index"]
    factor = row["share"]
    j = idx
    if idx not in costs_index:
        j = (idx[0], idx[1], idx[2], "World")
    try:
        sel = costs.loc[j]
    except KeyError:
        print(j)
        break
    sel["cost"] = factor * sel["cost"]
    sel = sel.pivot(index="quantile", columns="impact category", values="cost").reset_index()
    sel["REMIND index"] = tech
    sel["region"] = idx[-1]
    dflist.append(sel)
    
# return pd.concat(dflist, axis=0, ignore_index=True).groupby(["REMIND index", "region", "quantile"]).sum()

('hard coal, burned in hard coal industrial furnace 1-10MW', 'heat', 'megajoule', 'World')


In [ ]:
costs.index

MultiIndex([('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ('diesel, burned in diesel-electric generating set, 10MW', ...),
            ...
            (                    'natural gas, burned in gas turbine', ...),
            (                    'natural gas, burned in gas turbine', ...),
            (                    'natural gas, burned in gas

In [31]:
costs.loc[j]

KeyError: ('hard coal, burned in hard coal industrial furnace 1-10MW', 'heat', 'megajoule', 'World')

In [26]:
combine_shares_and_costs_new(regionalized_mapping, regionalized_costs)

ValueError: No objects to concatenate